# Scrapers for MyDramaList

You will follow the instructions in Part 4 of Week 2 Tasks. In the top part of the notebook summarize through a table of content what you decided to do and then explain why.

**Table of Contents**

1. [Scraping an aggregation page - top rated movies](#sec1)
2. [Scraping a dedicated page - train to busan movie](#sec2)
3. [Use pagination to scrape multiple pages - all pages from top rated sci-fi movies](#sec3)
4. [Use scrolling to scrape comments - squid game comments](#sec4)

**Why I decided to do these tasks**
<p>I chose these tasks because I have done some scraping before, but I have never scraped a page with dynamic content so it will be a challenge.</p>
<p>For the first two tasks, I will simply use requests and BeautifulSoup to extract the page content. For the final two tasks, I will use Selenium to work with the dynamic content.</p>
<p>Because I will be scraping multiple of the top movie pages for my part 3 pagination section, I can try to reuse my code from part 1 where I extract info from a single top movie aggregation page.</p>

<h2>Import requirements</h2>

In [12]:
import requests
from bs4 import BeautifulSoup
import re
import json
import math
from seleniumbase import Driver
from selenium.webdriver.common.by import By

<h2>Define common functions</h2>

In [6]:
def fetch_page_content(url):
    """Fetches HTML content from a URL and checks status code."""
    response = requests.get(url)
    if response.status_code == 200:
        return response.text
    else:
        return None

<h2 id="sec1">1. Scraping an aggregation page - top rated movies</h2>

<h3>Get page content and identify movie box cards</h3>

In [3]:
url = "https://mydramalist.com/movies/top"

html_content = fetch_page_content(url)

soup = BeautifulSoup(html_content, "html.parser")
movie_cards = soup.find_all(class_="box-body")

<h3>Extract individual movie content</h3>

<p>Define helpful function</p>

In [4]:
def get_single_movie_info(movie):
    movie_info = {}

    movie_info["rank"] = movie.find("div", class_="ranking pull-right").get_text(strip=True)[1:]

    movie_info["title"] = movie.find("h6").get_text(strip=True)

    movie_type_and_year = movie.find("span", class_="text-muted").get_text(strip=True)

    #create regex pattern to extract text  before dash seperator
    type_pattern = re.compile(r'^[^-]+(?=-)')
    movie_info["type"] = type_pattern.findall(movie_type_and_year)[0].strip()
    movie_info["release_year"] = movie_type_and_year.split()[-1]

    movie_info["rating"] = movie.find("span", class_="p-l-xs score").get_text(strip=True)

    #use boolean to indicate whether the movie has a streaming button
    movie_info["streaming"] = False
    movie_streaming = movie.find_all("a", class_="btn-watch-online btn-sm btn white")
    if movie_streaming:
        movie_info["streaming"] = True

    movie_synopsis = movie.find_all("p")[-1]
    movie_info["synopsis"] = movie_synopsis.get_text(strip=True)

    #get movie cover image from <img> data-src attribute
    movie_cover_image = movie.find_all("img")[0]
    movie_info["cover_image"] = movie_cover_image["data-src"]
    
    return movie_info

<p>Iterate through movies to save their info</p>

In [5]:
all_movies_info = []

for movie in movie_cards:
    try:
        movie_info = get_single_movie_info(movie)
        all_movies_info.append(movie_info)
    except Exception as e:
        pass

print(all_movies_info[0])

{'rank': '1', 'title': 'Better Days', 'type': 'Chinese Movie', 'release_year': '2019', 'rating': '9.2', 'streaming': True, 'synopsis': 'The film revolves around a girl who is being bullied at school and her relationship with a tough street kid, with whom she is implicated in the murder of a teenage girl. (Source: ScreenDaily) ~~ Adapted from the web…', 'cover_image': 'https://i.mydramalist.com/rmEw2s.jpg?v=1'}


<p>Save to JSON file</p>

In [6]:
with open("top_rated_movies.json", mode="w", encoding="utf-8") as write_file:
    json.dump(all_movies_info, write_file, indent = 2)

<h2 id="sec2">2. Scraping a dedicated page (train to busan movie)</h2>

<h3>Get page content and identify movie box cards</h3>

In [7]:
url = "https://mydramalist.com/11314-train-to-busan"

html_content = fetch_page_content(url)

soup = BeautifulSoup(html_content, "html.parser")
page = soup.find("body")

<h3>Extract dedicated page movie content</h3>

In [8]:
movie_info = {}

#get the movie card box
movie_card = page.find("div", class_="box")

movie_title = movie_card.find("h1", class_="film-title").get_text(strip=True)
movie_info["title"] = movie_title.split("(")[0].strip()

#get the first info list
list_one = movie_card.find("ul", class_="list m-a-0")
list_one_items = list_one.find_all("li")

#get the second info list
list_two = movie_card.find("ul", class_="list m-a-0 hidden-md-up")
list_two_items = list_two.find_all("li")

movie_info["native_title"] = list_one_items[1].find("a").get_text(strip=True)

movie_info["release_date"] = list_two_items[2].get_text(strip=True).split(":")[-1]
movie_info["country"] = list_two_items[0].get_text(strip=True).split(":")[-1]
movie_info["type"] = list_two_items[1].get_text(strip=True).split(":")[-1]
movie_info["duration"] = list_two_items[3].get_text(strip=True).split(":")[-1]

movie_info["rating"] = list_two_items[4].get_text(strip=True).split(":")[-1].split("(")[0]
movie_info["ranking"] = list_two_items[5].get_text(strip=True).split(":")[-1]
movie_info["popularity"] = list_two_items[6].get_text(strip=True).split(":")[-1]

movie_info["director"] = list_one_items[3].get_text(strip=True).split(":")[-1]
movie_info["screenwriter"] = list_one_items[4].get_text(strip=True).split(":")[-1]

#get each service from the 'where to stream' box
streaming_info = page.find_all("div", class_="p-l")

where_to_watch = []
for service in streaming_info:
    where_to_watch.append(service.find("b").get_text(strip=True))
movie_info["where_to_watch"] = where_to_watch

#get each name (link) from the cast box
cast_info = page.find_all("a", class_="text-primary text-ellipsis")

cast = []
for person in cast_info:
    cast.append(person.find("b").get_text(strip=True))
movie_info["cast"] = cast

movie_info["synopsis"] = movie_card.find("div", class_="show-synopsis").get_text(strip=True).split("\n")[0]

#get movie cover image link from <img> src attribute
movie_cover_image = movie_card.find("img", class_="img-responsive")
movie_info["cover_image"] = movie_cover_image["src"]

print(movie_info)

{'title': 'Train to Busan', 'native_title': '부산행', 'release_date': 'Jul 20, 2016', 'country': 'South Korea', 'type': 'Movie', 'duration': '1 hr. 58 min.', 'rating': '8.9', 'ranking': '#87', 'popularity': '#32', 'director': 'Yeon Sang Ho', 'screenwriter': 'Park Joo Seok', 'where_to_watch': ['Apple TV', 'Netflix', 'Viki', 'Prime Video', 'Tubi'], 'cast': ['Gong Yoo', 'Jung Yu Mi', 'Ma Dong Seok', 'Kim Soo An', 'Kim Eui Sung', 'Choi  Woo Shik'], 'synopsis': 'Seok Woo, his estranged daughter Soo An, and other passengers become trapped on a KTX train (high-speed train) heading from Seoul to Busan during a disastrous virus outbreak in South Korea.', 'cover_image': 'https://i.mydramalist.com/xklBrc.jpg?v=1'}


<p>Save to JSON file</p>

In [9]:
with open("train_to_busan_movie_info.json", mode="w", encoding="utf-8") as write_file:
    json.dump(movie_info, write_file, indent = 2)

<h2 id=#sec3">3. Use pagination to scrape multiple pages - all pages from top rated sci-fi movies</h2>

<h3>Get page content and identify the number of pages</h3>

In [33]:
url = "https://mydramalist.com/search?adv=titles&ty=77&ge=27&re=1890,2026&rt=1,10&so=rated"

html_content = fetch_page_content(url)

soup = BeautifulSoup(html_content, "html.parser")

results = soup.find("p", class_="m-b-sm pull-right").get_text(strip=True).split(" ")[0]
results = int(results)

pages = math.ceil(results/20)

print(f"{results} results")
print(f"{pages} pages")

252 results
13 pages


<h3>Extract movie content across multiple pages</h3>

In [34]:
all_pages_info = []

#i have to use firefox, when I use chrome it doesn't work
with Driver(browser="firefox") as driver:
    for page_number in range(1, pages + 1):
        page_url = f"{url}&page={page_number}"
        driver.open(page_url)
        driver.sleep(2.0)

        rendered_html = driver.get_page_source()
        soup = BeautifulSoup(rendered_html, "html.parser")
        movie_cards = soup.find_all(class_="box-body")

        for movie in movie_cards:
            try:
                movie_info = get_single_movie_info(movie)
                movie_info["page_number"] = page_number
                all_pages_info.append(movie_info)
            except Exception as error:
                pass

print(len(all_pages_info))
print(all_pages_info[0])

252
{'rank': '87', 'title': 'Train to Busan', 'type': 'Korean Movie', 'release_year': '2016', 'rating': '8.9', 'streaming': True, 'synopsis': 'Seok Woo, his estranged daughter Soo An, and other passengers become trapped on a KTX train (high-speed train) heading from Seoul to Busan during a disastrous virus outbreak in South Korea.\n\n(Source: MyDramaList)', 'cover_image': 'https://i.mydramalist.com/xklBrs.jpg?v=1', 'page_number': 1}


<p>Save to JSON file</p>

In [35]:
with open("top_rated_sci_fi_movies_all_pages.json", mode="w", encoding="utf-8") as write_file:
    json.dump(all_pages_info, write_file, indent = 2)

<h2 id="sec4">4. Use scrolling to scrape comments - squid game comments</h2>

In [ ]:
def scrape_one_comment(html_content):
    return

In [ ]:
last_count = 0

with Driver(browser="firefox") as driver:
    driver.open(url)

    while True:
        driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")
        driver.sleep(1.0)
        
        current_count = len(driver.find_elements(By.CLASS_NAME, "comment"))
        print(f"Drama count: {current_count}")
        
        if current_count == last_count:
            button_exists = len(driver.find_elements(By.XPATH, "//*[contains(text(), 'Load more comments')]"))
            if button_exists == 1:
                driver.click("Load more comments")
                driver.sleep(1)
            else:
                pass
        last_count = current_count

Drama count: 76
Drama count: 76
1


NoSuchElementException: Message: 
 Element {Load more comments} was not present after 7 seconds!
